# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Haryomhidhe/Machine-learning-flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"SELECT DISTINCT regexp_extract(file, 'month=[0-9-]+') AS month FROM glob('{rel}/fact_content_daily_performance/*/*.parquet') ORDER BY month").show()

┌───────────────┐
│     month     │
│    varchar    │
├───────────────┤
│ month=2025-01 │
│ month=2025-02 │
│ month=2025-03 │
│ month=2025-04 │
│ month=2025-05 │
│ month=2025-06 │
│ month=2025-07 │
│ month=2025-08 │
│ month=2025-09 │
│ month=2025-10 │
│ month=2025-11 │
│ month=2025-12 │
│ month=2026-01 │
│ month=2026-02 │
│ month=2026-03 │
│ month=2026-04 │
│ month=2026-05 │
│ month=2026-06 │
├───────────────┤
│    18 rows    │
└───────────────┘



In [18]:
feb = con.sql(f"""
SELECT content_hash_id,
  SUM(gsc_impressions) AS imps,
  SUM(gsc_clicks) AS clicks,
  AVG(gsc_avg_position) AS pos
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

base = feb[feb['imps'] > 100].copy()
base['pos_bucket'] = pd.cut(base['pos'], [0, 2, 3, 5, 10, 20, 50, 10000],
                            labels=['1-2', '2-3', '3-5', '5-10', '10-20', '20-50', '50+'])
expected = base.groupby('pos_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    clicks=('clicks', 'sum'),
    imps=('imps', 'sum'))
expected['expected_ctr_pct'] = expected['clicks'] / expected['imps'] * 100
expected

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,clicks,imps,expected_ctr_pct
pos_bucket,,,,
1-2,4839,41847.0,11866419.0,0.352651
2-3,6235,94953.0,21542278.0,0.440775
3-5,14591,189199.0,51589704.0,0.366738
5-10,26394,154377.0,50888000.0,0.303366
10-20,16706,69201.0,22252028.0,0.310987
20-50,10302,30317.0,19883322.0,0.152475
50+,1056,174.0,366912.0,0.047423


In [19]:
base['ctr'] = base['clicks'] / base['imps'] * 100
base.groupby('pos_bucket', observed=True)['ctr'].agg(
    n='count', median_ctr='median', p75=lambda s: s.quantile(0.75))

,n,median_ctr,p75
pos_bucket,,,
1-2,4839,0.161812,0.376530
2-3,6235,0.248822,0.509431
3-5,14591,0.224803,0.447761
5-10,26394,0.160508,0.402901
10-20,16706,0.073046,0.335219
20-50,10302,0.000000,0.170322
50+,1056,0.000000,0.000000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.